# t-SNE: Visual Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/methods-algorithms/blob/master/notebooks/L05_tsne.ipynb)

**Learning Objectives:**
- See how t-SNE reveals clusters that PCA cannot separate
- Understand the effect of perplexity on the embedding
- Learn what t-SNE distances do and do not mean
- Compare t-SNE with PCA and UMAP on real and synthetic data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import load_digits, make_classification
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import time

np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# ML color palette
ML_PURPLE = '#3333B2'
ML_BLUE = '#0066CC'
ML_ORANGE = '#FF7F0E'
ML_GREEN = '#2CA02C'
ML_RED = '#D62728'

# 10-class colormap for digits
DIGIT_COLORS = plt.cm.tab10(np.linspace(0, 1, 10))

print('Setup complete.')

## 1. Load High-Dimensional Data

In [ ]:
# Load digits dataset (64 features = 8x8 pixel images, 10 classes)
digits = load_digits()
X, y = digits.data, digits.target

print(f'Shape: {X.shape} ({X.shape[0]} samples, {X.shape[1]} features)')
print(f'Classes: {np.unique(y)} (digits 0-9)')
print(f'Samples per class: {np.bincount(y)}')

# Visual 1: Show 10 sample digit images (2x5 grid)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    idx = np.where(y == i)[0][0]  # first sample of each digit
    ax.imshow(digits.images[idx], cmap='gray_r', interpolation='nearest')
    ax.set_title(f'Digit {i}', fontsize=12)
    ax.axis('off')

plt.suptitle('Sample Digit Images (8x8 pixels = 64 features)', fontsize=14)
plt.tight_layout()
plt.show()

## 2. PCA First -- Does It Separate Classes?

In [ ]:
# Visual 2: PCA(n_components=2) on digits
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 6))
for digit in range(10):
    mask = y == digit
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=15, alpha=0.5,
               color=DIGIT_COLORS[digit], label=str(digit))

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA Projection of Digits (64D -> 2D) -- Classes Overlap')
ax.legend(title='Digit', fontsize=8, ncol=2, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. t-SNE Reveals Hidden Clusters

In [ ]:
# Visual 3: t-SNE on digits
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 6))
for digit in range(10):
    mask = y == digit
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=15, alpha=0.6,
               color=DIGIT_COLORS[digit], label=str(digit))

ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('t-SNE Projection of Digits (64D -> 2D) -- Clear Clusters!')
ax.legend(title='Digit', fontsize=8, ncol=2, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visual 4: Side-by-side PCA vs t-SNE
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for digit in range(10):
    mask = y == digit
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], s=10, alpha=0.5,
                    color=DIGIT_COLORS[digit], label=str(digit))
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=10, alpha=0.5,
                    color=DIGIT_COLORS[digit], label=str(digit))

axes[0].set_title('PCA (linear) -- Classes Overlap', fontsize=13)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].grid(True, alpha=0.3)

axes[1].set_title('t-SNE (non-linear) -- Clear Separation', fontsize=13)
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].grid(True, alpha=0.3)

# Shared legend
handles = [mpatches.Patch(color=DIGIT_COLORS[d], label=str(d)) for d in range(10)]
fig.legend(handles=handles, title='Digit', loc='center right', fontsize=9)

plt.suptitle('PCA vs t-SNE on Digits (1797 samples, 64 features)', fontsize=14)
plt.tight_layout(rect=[0, 0, 0.92, 0.96])
plt.show()

## 4. Effect of Perplexity

In [ ]:
# Visual 5: 3-panel perplexity comparison (use subset for speed)
np.random.seed(42)
subset_idx = np.random.choice(len(X), size=500, replace=False)
X_sub = X_scaled[subset_idx]
y_sub = y[subset_idx]

perplexities = [5, 30, 100]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, perp in zip(axes, perplexities):
    emb = TSNE(n_components=2, perplexity=perp, random_state=42,
               n_iter=1000).fit_transform(X_sub)
    for digit in range(10):
        mask = y_sub == digit
        ax.scatter(emb[mask, 0], emb[mask, 1], s=15, alpha=0.6,
                   color=DIGIT_COLORS[digit])
    ax.set_title(f'Perplexity = {perp}', fontsize=13)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(True, alpha=0.3)

plt.suptitle('Effect of Perplexity on t-SNE (500 digits)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. What t-SNE Distances Mean (and Don't Mean)

In [ ]:
# Visual 6: Trustworthiness and silhouette for different perplexities
from sklearn.manifold import trustworthiness

perp_range = [5, 10, 20, 30, 50, 100]
trust_scores = []
sil_scores = []

for perp in perp_range:
    emb = TSNE(n_components=2, perplexity=perp, random_state=42,
               n_iter=1000).fit_transform(X_sub)
    trust = trustworthiness(X_sub, emb, n_neighbors=15)
    sil = silhouette_score(emb, y_sub)
    trust_scores.append(trust)
    sil_scores.append(sil)
    print(f'Perplexity={perp:3d}: trustworthiness={trust:.3f}, silhouette={sil:.3f}')

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

l1 = ax1.plot(perp_range, trust_scores, 'o-', color=ML_BLUE, linewidth=2,
              markersize=8, label='Trustworthiness')
l2 = ax2.plot(perp_range, sil_scores, 's-', color=ML_ORANGE, linewidth=2,
              markersize=8, label='Silhouette Score')

ax1.set_xlabel('Perplexity')
ax1.set_ylabel('Trustworthiness', color=ML_BLUE)
ax2.set_ylabel('Silhouette Score', color=ML_ORANGE)
ax1.set_title('Embedding Quality vs. Perplexity')
ax1.grid(True, alpha=0.3)

lines = l1 + l2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='lower right')

plt.tight_layout()
plt.show()

## 6. t-SNE on a Simple Dataset

In [ ]:
# Visual 7: t-SNE on synthetic 20-feature, 4-class data
X_synth, y_synth = make_classification(
    n_samples=400, n_features=20, n_informative=5,
    n_redundant=5, n_classes=4, n_clusters_per_class=1,
    random_state=42
)
X_synth_scaled = StandardScaler().fit_transform(X_synth)

tsne_synth = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_synth_tsne = tsne_synth.fit_transform(X_synth_scaled)

synth_colors = [ML_BLUE, ML_ORANGE, ML_GREEN, ML_RED]

fig, ax = plt.subplots(figsize=(10, 6))
for k in range(4):
    mask = y_synth == k
    ax.scatter(X_synth_tsne[mask, 0], X_synth_tsne[mask, 1], s=40, alpha=0.6,
               color=synth_colors[k], label=f'Class {k}', edgecolors='white')

ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('t-SNE on Synthetic Data (20 features, 4 classes)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. UMAP Comparison

In [ ]:
# Visual 8: Try UMAP if available, otherwise compare PCA vs t-SNE timing
try:
    import umap
    has_umap = True
except ImportError:
    has_umap = False
    print('umap-learn not installed. Comparing PCA vs t-SNE timing instead.')

if has_umap:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # t-SNE
    t0 = time.time()
    emb_tsne = TSNE(n_components=2, perplexity=30, random_state=42,
                    n_iter=1000).fit_transform(X_scaled)
    t_tsne = time.time() - t0

    # UMAP
    t0 = time.time()
    reducer = umap.UMAP(n_components=2, random_state=42)
    emb_umap = reducer.fit_transform(X_scaled)
    t_umap = time.time() - t0

    for digit in range(10):
        mask = y == digit
        axes[0].scatter(emb_tsne[mask, 0], emb_tsne[mask, 1], s=10, alpha=0.5,
                        color=DIGIT_COLORS[digit])
        axes[1].scatter(emb_umap[mask, 0], emb_umap[mask, 1], s=10, alpha=0.5,
                        color=DIGIT_COLORS[digit])

    axes[0].set_title(f't-SNE ({t_tsne:.1f}s)', fontsize=13)
    axes[1].set_title(f'UMAP ({t_umap:.1f}s)', fontsize=13)
    for ax in axes:
        ax.grid(True, alpha=0.3)

    plt.suptitle('t-SNE vs UMAP on Digits (1797 samples)', fontsize=14)
    plt.tight_layout()
    plt.show()
    print(f't-SNE: {t_tsne:.1f}s, UMAP: {t_umap:.1f}s')

else:
    # Timing comparison: PCA vs t-SNE
    t0 = time.time()
    _ = PCA(n_components=2).fit_transform(X_scaled)
    t_pca = time.time() - t0

    t0 = time.time()
    _ = TSNE(n_components=2, perplexity=30, random_state=42,
             n_iter=1000).fit_transform(X_scaled)
    t_tsne = time.time() - t0

    fig, ax = plt.subplots(figsize=(10, 6))
    methods = ['PCA', 't-SNE']
    times = [t_pca, t_tsne]
    colors = [ML_BLUE, ML_ORANGE]
    bars = ax.bar(methods, times, color=colors, alpha=0.7, edgecolor='white', linewidth=1.5)

    for bar, t in zip(bars, times):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f'{t:.2f}s', ha='center', fontsize=12, fontweight='bold')

    ax.set_ylabel('Time (seconds)')
    ax.set_title(f'PCA vs t-SNE Runtime (n={len(X)}, p={X.shape[1]})')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

    speedup = t_tsne / max(t_pca, 1e-6)
    print(f'PCA: {t_pca:.3f}s, t-SNE: {t_tsne:.1f}s ({speedup:.0f}x slower)')
    print('Install umap-learn (`pip install umap-learn`) for UMAP comparison.')

## Summary

**Key Takeaways:**
- **t-SNE reveals clusters invisible to PCA** by preserving local neighborhood structure rather than global variance
- **Perplexity controls the local/global tradeoff**: low perplexity shows fine detail, high perplexity shows broader structure -- always try 3+ values
- **Do not over-interpret t-SNE plots**: cluster sizes, inter-cluster distances, and axis orientation are all meaningless
- **UMAP is a modern alternative** that is faster, preserves more global structure, and supports projecting new points